# Itinerary Generator Pipeline — Step-by-Step

Walks through `services/trip-generation/` one phase at a time. Run cells
top-to-bottom the first time; afterwards you can tweak any intermediate
variable and re-run just the downstream phase.

**Phases**
1. Route search & composition
2. POI retrieval + enrichment + union with route POIs
3. LLM pick (Gemini)
4. Solver — order stops, enforce opening hours & meal anchors
5. Persist (optional — writes to Supabase)

**Before you start:** see `notebooks/README.md` for tslab install. Launch
`jupyter lab` from the **repo root**, not from `notebooks/`.

## Setup — load `.env.local` and sanity-check keys

In [1]:
import { dirname, resolve } from "node:path";
import { existsSync } from "node:fs";

let repoRoot = process.cwd();
while (!existsSync(resolve(repoRoot, ".env.local")) && dirname(repoRoot) !== repoRoot) {
  repoRoot = dirname(repoRoot);
}

const envPath = resolve(repoRoot, ".env.local");
if (!existsSync(envPath)) {
  throw new Error(`Missing .env.local above ${process.cwd()}. Did you start jupyter inside this repo?`);
}

if (process.cwd() !== repoRoot) {
  process.chdir(repoRoot);
}
import "tsx/cjs";
process.loadEnvFile(envPath);

const required = [
  "NEXT_PUBLIC_SUPABASE_URL",
  "SUPABASE_SECRET_KEY",
  "GEMINI_API_KEY",
];
const missing = required.filter((k) => !process.env[k]);
if (missing.length) throw new Error(`Missing env vars: ${missing.join(", ")}`);

console.log("repo root:", repoRoot);
console.log("env loaded:", required.map((k) => `${k}=${process.env[k]?.slice(0, 8)}...`).join("  "));
console.log("GOOGLE_PLACES_API_KEY:", process.env.GOOGLE_PLACES_API_KEY ? "set" : "(missing - enrichment will be skipped)");


repo root: E:\Projects\travel-sync-ai
env loaded: NEXT_PUBLIC_SUPABASE_URL=https://...  SUPABASE_SECRET_KEY=sb_secre...  GEMINI_API_KEY=AIzaSyCx...
GOOGLE_PLACES_API_KEY: set


## Imports — pipeline functions

Path alias `@/*` is wired through `notebooks/tsconfig.json` so we can use
the same imports the production code does.

In [2]:
const routeEngine = require("@/services/trip-generation/route-engine");
const poiEngine = require("@/services/trip-generation/poi-engine");
const solver = require("@/services/trip-generation/solver");
const orchestrator = require("@/services/trip-generation/orchestrator");
const { randomBytes } = require("node:crypto");

Object.assign(globalThis, {
  searchRoutesByVibe: routeEngine.searchRoutesByVibe,
  composeFromRoutes: routeEngine.composeFromRoutes,
  searchPoisByVibe: poiEngine.searchPoisByVibe,
  loadPoisByIds: poiEngine.loadPoisByIds,
  enrichWithLiveData: poiEngine.enrichWithLiveData,
  solveItinerary: solver.solveItinerary,
  PACE_CAPS: solver.PACE_CAPS,
  __notebook: orchestrator.__notebook,
});

const genId = randomBytes(4).toString("hex");
(globalThis as any).genId = genId;
console.log("genId for this notebook run:", genId);


genId for this notebook run: 1559d757


## Pre-flight — load curated Niseko data and seed routes

Phase 1a needs `route_templates` rows for the destination (with matching
pace + alias) and Phase 2b needs the route's `place_ids` to resolve in
`poi_embeddings` or the health filter drops them.

This cell idempotently seeds the example trip using the **curated dataset
under `data/japan-ski-trip/niseko/`** (resorts, restaurants, activities).
It builds two multi-stop balanced-pace routes — Hirafu day and Niseko
Village day — that each compose a complete ski-day-out: morning ski →
lunch → afternoon onsen → dinner. Restaurant opening hours are pinned to
the solver's meal-anchor windows (lunch opens 12:00, dinner opens 18:00)
so the curator-ordered routes are guaranteed feasible.

If the routes already exist for this destination, the cell short-circuits
into a no-op. If you've already run `scripts/ingest-ski-dataset.ts` +
`scripts/seed-route-templates.ts`, those rows live under
`Hokkaido, Japan` and don't collide with this `Niseko, Japan` seed.

In [ ]:
import { readFileSync } from "node:fs";
import { resolve as _resolve } from "node:path";
import { createAdminClient } from "@/lib/db";
import { generateEmbedding as _genEmb } from "@/lib/gemini";
import { normalizeAliases } from "@/lib/destination-aliases";

const _SEED_DEST = "Niseko, Japan";
const _SEED_ALIASES = normalizeAliases([
  "Niseko, Japan", "Niseko", "ニセコ",
  "Hokkaido, Japan", "Hokkaido", "北海道",
  "Japan", "Japan ski", "ski japan",
]);

// Curated dataset shipped in the repo (no network calls).
const _dataDir = _resolve(process.cwd(), "data/japan-ski-trip/niseko");
const _resortsRaw = JSON.parse(readFileSync(_resolve(_dataDir, "resorts.json"), "utf8"));
const _restaurantsRaw = JSON.parse(readFileSync(_resolve(_dataDir, "restaurants.json"), "utf8"));
const _activitiesRaw = JSON.parse(readFileSync(_resolve(_dataDir, "activities.json"), "utf8"));

interface _Resort { id: string; name: string; name_ja: string; lat: number; lng: number; night_skiing: boolean; notes: string }
interface _Restaurant { id: string; name: string; cuisine: string; lat: number; lng: number; notes: string }
interface _Activity { id: string; name: string; type: string; lat: number; lng: number; notes: string }

const _resorts = _resortsRaw.resorts as _Resort[];
const _restaurants = _restaurantsRaw.restaurants as _Restaurant[];
const _activities = _activitiesRaw.activities as _Activity[];

// 7-day-a-week opening periods.
const _allDays = (open: number, close: number) =>
  Array.from({ length: 7 }, (_, d) => ({ openDay: d, openMinutes: open, closeDay: d, closeMinutes: close }));

const _resortPeriods = (nightSki: boolean) => {
  const days = _allDays(8*60+30, 16*60+30);
  if (nightSki) days.push(..._allDays(17*60, 20*60+30));
  return days;
};
// Restaurant hours pinned to the solver's meal anchors. Real Niseko venues
// typically open 11:30 / 17:30, but the solver enforces arrive ∈ [12:00,14:00]
// for lunch and [18:00,20:00] for dinner — opening exactly on those edges
// lets the curator order [activity, restaurant] without manual padding stops.
const _LUNCH_PERIODS = _allDays(12*60, 15*60);
const _DINNER_PERIODS = _allDays(18*60, 22*60);
const _ONSEN_PERIODS = _allDays(10*60, 22*60);

// Multi-stop balanced-pace routes; preorderedDays=true keeps the solver from
// re-permuting them, so the listed order *is* the schedule.
const _ROUTES: Array<{ title: string; summary: string; place_ids: string[] }> = [
  {
    title: "Hirafu 滑雪一日",
    summary:
      "Niseko Grand Hirafu 早場滑雪、Tsubara Tsubara 湯咖哩午餐、街中 Yukoro 溫泉、Bang Bang 串燒晚餐。" +
      "Best-known Niseko United base + classic apres-ski walking circuit, all within Hirafu Village.",
    place_ids: ["niseko-grand-hirafu", "tsubara-tsubara", "yukoro-onsen", "bang-bang"],
  },
  {
    title: "Niseko Village 寬鬆滑雪日",
    summary:
      "Niseko Village 滑雪、Rakuichi 手打蕎麥午餐、昆布溫泉 Tsuruga 露天浴、Ezo Seafoods 北海道海鮮晚餐。" +
      "Quieter side of Niseko United for relaxed couples; ends in an onsen and oysters.",
    place_ids: ["niseko-village", "rakuichi-soba", "niseko-konbu-onsen-tsuruga", "ezo-seafoods"],
  },
];

const _LUNCH_IDS = new Set(["tsubara-tsubara", "rakuichi-soba", "graubunden"]);
const _DINNER_IDS = new Set(["bang-bang", "ezo-seafoods", "abucha-2", "the-barn-by-odin"]);

const _db = createAdminClient();

// No-op when both routes already present.
const { data: _existingRoutes } = await _db
  .from("route_templates")
  .select("title")
  .eq("destination_name", _SEED_DEST)
  .eq("is_archived", false);
const _haveTitles = new Set((_existingRoutes ?? []).map((r) => r.title));
const _missingRoutes = _ROUTES.filter((r) => !_haveTitles.has(r.title));

if (_missingRoutes.length === 0) {
  console.log(`pre-flight: routes already present for ${_SEED_DEST}, skipping`);
} else {
  // Only seed the POIs referenced by the missing routes.
  const _neededPoiIds = new Set<string>();
  for (const r of _missingRoutes) for (const id of r.place_ids) _neededPoiIds.add(id);

  type _Row = {
    place_id: string; destination_name: string; destination_aliases: string[];
    name: string; item_type: string; tags: string[]; description: string;
    lat: number; lng: number; live_data: object; _embedText: string;
  };
  const _poiRows: _Row[] = [];

  for (const r of _resorts) {
    if (!_neededPoiIds.has(r.id)) continue;
    _poiRows.push({
      place_id: r.id,
      destination_name: _SEED_DEST,
      destination_aliases: normalizeAliases([..._SEED_ALIASES, r.id, r.name, r.name_ja]),
      name: r.name,
      item_type: "activity",
      tags: ["ski", "snow", "winter", "adventure", ...(r.night_skiing ? ["nightlife"] : [])],
      description: r.notes,
      lat: r.lat,
      lng: r.lng,
      live_data: {
        placeId: r.id, name: r.name, address: null, rating: null, priceLevel: null,
        lat: r.lat, lng: r.lng, openingPeriods: _resortPeriods(r.night_skiing),
      },
      _embedText: `Ski resort in ${_SEED_DEST}. ${r.name} (${r.name_ja}). ${r.notes}`,
    });
  }
  for (const rt of _restaurants) {
    if (!_neededPoiIds.has(rt.id)) continue;
    const isLunch = _LUNCH_IDS.has(rt.id);
    const isDinner = _DINNER_IDS.has(rt.id);
    if (!isLunch && !isDinner) {
      throw new Error(`restaurant ${rt.id} is not classified as lunch or dinner; classify in _LUNCH_IDS/_DINNER_IDS`);
    }
    _poiRows.push({
      place_id: rt.id,
      destination_name: _SEED_DEST,
      destination_aliases: normalizeAliases([..._SEED_ALIASES, rt.id, rt.name]),
      name: rt.name,
      item_type: "restaurant",
      tags: ["food", rt.cuisine, isLunch ? "lunch" : "dinner"],
      description: rt.notes,
      lat: rt.lat,
      lng: rt.lng,
      live_data: {
        placeId: rt.id, name: rt.name, address: null, rating: null, priceLevel: null,
        lat: rt.lat, lng: rt.lng,
        openingPeriods: isLunch ? _LUNCH_PERIODS : _DINNER_PERIODS,
      },
      _embedText: `Restaurant in ${_SEED_DEST}. ${rt.name} (${rt.cuisine}). ${rt.notes}`,
    });
  }
  for (const a of _activities) {
    if (!_neededPoiIds.has(a.id)) continue;
    _poiRows.push({
      place_id: a.id,
      destination_name: _SEED_DEST,
      destination_aliases: normalizeAliases([..._SEED_ALIASES, a.id, a.name]),
      name: a.name,
      item_type: "activity",
      tags: ["onsen", "relaxed", "winter"],
      description: a.notes,
      lat: a.lat,
      lng: a.lng,
      live_data: {
        placeId: a.id, name: a.name, address: null, rating: null, priceLevel: null,
        lat: a.lat, lng: a.lng, openingPeriods: _ONSEN_PERIODS,
      },
      _embedText: `Onsen experience in ${_SEED_DEST}. ${a.name}. ${a.notes}`,
    });
  }

  const _haveIds = new Set(_poiRows.map((r) => r.place_id));
  const _missingIds = Array.from(_neededPoiIds).filter((id) => !_haveIds.has(id));
  if (_missingIds.length) {
    throw new Error(`pre-flight: no curated record for ${_missingIds.join(", ")}`);
  }

  console.log(`pre-flight: seeding ${_poiRows.length} POIs + ${_missingRoutes.length} routes for ${_SEED_DEST}`);

  for (const p of _poiRows) {
    const embedding = await _genEmb(p._embedText);
    const { _embedText: _omit, ...rest } = p;
    const { error } = await _db.from("poi_embeddings").upsert(
      { ...rest, embedding, source: "notebook-seed", last_seen_at: new Date().toISOString() },
      { onConflict: "place_id" }
    );
    if (error) throw new Error(`poi upsert failed for ${p.place_id}: ${error.message}`);
    console.log(`  + poi ${p.place_id} (${p.item_type})`);
  }

  for (const r of _missingRoutes) {
    const text =
      `Travel experiences in ${_SEED_DEST} for a balanced-paced, mid-budget trip ` +
      `with a ski, snow, onsen, winter vibe. ${r.title}. ${r.summary}`;
    const embedding = await _genEmb(text);
    const { error } = await _db.from("route_templates").upsert(
      {
        destination_name: _SEED_DEST,
        destination_aliases: _SEED_ALIASES,
        title: r.title,
        summary: r.summary,
        vibe_tags: ["ski", "snow", "winter", "onsen", "relaxed", "food"],
        pace: "balanced",
        place_ids: r.place_ids,
        pinned_vibes: ["ski"],
        embedding,
        source: "notebook-seed",
        last_health_ok_at: new Date().toISOString(),
      },
      { onConflict: "destination_name,title", ignoreDuplicates: false }
    );
    if (error) throw new Error(`route upsert failed for ${r.title}: ${error.message}`);
    console.log(`  + route "${r.title}" (${r.place_ids.length} stops)`);
  }
  console.log("pre-flight: done");
}


## Phase 0 — Survey input

Edit this cell to change the destination, party, vibe, etc., then re-run
everything below. The defaults below match the pre-flight seed (Niseko,
2-day couple ski/snow/onsen, balanced pace) so Phase 1a clears the 0.72
gate and Phase 4 produces a feasible 4-stop day for each of the two
days. If you change the destination, make sure the corresponding
`route_templates` + `poi_embeddings` rows exist — otherwise Phase 1a
returns 0 and Phase 2a falls back to Google Places.

In [3]:
const answers: SurveyAnswers = {
  destination: "Niseko, Japan",
  duration_days: 2,
  party: "couple",
  party_size: 2,
  budget_tier: "mid",
  vibe: ["ski", "snow", "onsen"],
  pace: "balanced",
  must_haves: null,
};

const input: GenerateInput = {
  answers,
  authorLineUserId: "U_notebook_dev",
  startDate: undefined, // defaults to today + 14d
};

__notebook.validateAnswers(input);
const startWeekday = __notebook.deriveStartWeekday(input.startDate);
console.log("validated. startWeekday =", startWeekday, " (0=Sun … 6=Sat)");


## Phase 1a — Search curated routes

Vector-search `route_templates` by vibe. Returns up to 10 candidate
single-day routes, scored on similarity + boost + quality + pinned vibes.
Gate is 0.72 — anything below is filtered out.

In [4]:
const routes = await searchRoutesByVibe({
  destination: answers.destination!,
  vibe: answers.vibe,
  pace: answers.pace,
  budget: answers.budget_tier,
  k: 10,
  genId,
});

console.log(`routes found: ${routes.length}`);
console.table(routes.slice(0, 10).map((r) => ({
  routeId: r.routeId.slice(0, 8),
  title: r.title,
  score: r.finalScore.toFixed(3),
  similarity: r.similarity.toFixed(3),
  places: r.placeIds.length,
})));


{"level":"info","msg":"[route-engine] no rows from RPC","ts":"2026-05-26T04:32:00.898Z","genId":"1559d757","destination":"Kyoto","pace":"balanced"}
routes found: 0
┌─────────┐
│ (index) │
├─────────┤
└─────────┘


## Phase 1b — Compose routes into day-slots

Greedy packer assigns routes to days, avoiding place_id collisions. Any
day that isn't covered drops into `uncoveredDays`, which Phase 3 (LLM)
will fill.

In [5]:
const compose = composeFromRoutes(routes, answers.duration_days!);

console.log("covered days:");
for (const [day, route] of compose.coveredDays) {
  console.log(`  D${day}  ${route.title}  (${route.placeIds.length} stops)`);
}
console.log("uncovered days (LLM will pick for these):", compose.uncoveredDays);
console.log("place_ids reserved by routes:", compose.usedPlaceIds.size);


covered days:
uncovered days (LLM will pick for these): [ 1, 2, 3 ]
place_ids reserved by routes: 0


## Phase 2a — Retrieve POI candidates

Vector ANN search on `poi_embeddings`. Falls back to Google Places text
search if the corpus is cold for this destination.

In [6]:
const poiCandidates = await searchPoisByVibe({
  destination: answers.destination!,
  vibe: answers.vibe,
  pace: answers.pace,
  budget: answers.budget_tier,
  k: 30,
  genId,
});

console.log(`POI candidates: ${poiCandidates.length}`);
console.table(poiCandidates.slice(0, 15).map((p) => ({
  name: p.name,
  type: p.itemType,
  similarity: p.similarity.toFixed(3),
  tags: (p.tags ?? []).slice(0, 4).join(","),
})));


{"level":"warn","msg":"[poi-engine] vector search returned 0 rows, falling back","ts":"2026-05-26T04:32:01.224Z","genId":"1559d757","destination":"Kyoto"}


{"level":"info","msg":"[poi-engine] live text fallback","ts":"2026-05-26T04:32:02.271Z","genId":"1559d757","destination":"Kyoto","buckets":"activity,restaurant,hotel","returned":30}
POI candidates: 30
┌─────────┬─────────────────────────────────────────────────────────────┬──────────────┬────────────┬──────┐
│ (index) │ name                                                        │ type         │ similarity │ tags │
├─────────┼─────────────────────────────────────────────────────────────┼──────────────┼────────────┼──────┤
│ 0       │ 'Kinkaku-ji'                                                │ 'activity'   │ '0.500'    │ ''   │
│ 1       │ '伏見稻荷大社'                                              │ 'activity'   │ '0.500'    │ ''   │
│ 2       │ '二條城'                                                    │ 'activity'   │ '0.500'    │ ''   │
│ 3       │ 'SAMURAI NINJA MUSEUM Kyoto'                                │ 'activity'   │ '0.500'    │ ''   │
│ 4       │ '清水寺'                            

## Phase 2b — Union with route POIs

Routes lock specific place_ids; we materialize them as POI candidates
and merge into the shortlist. Route POIs take precedence.

In [7]:
const routePois = await loadPoisByIds(Array.from(compose.usedPlaceIds), genId);
const allCandidates = __notebook.unionByPlaceId(routePois, poiCandidates);

console.log(`route POIs: ${routePois.length} | search POIs: ${poiCandidates.length} | unioned: ${allCandidates.length}`);
if (allCandidates.length === 0) throw new Error("no candidates — pipeline would abort with no_candidates");


route POIs: 0 | search POIs: 30 | unioned: 30


## Phase 2c — Enrich with live data (Google Places)

Batch-fetches address, coords, and opening periods. The solver requires
coords + opening hours; without them, a POI is effectively unschedulable.

In [8]:
const enriched = await enrichWithLiveData(allCandidates);

const withCoords = enriched.filter((e) => e.lat != null && e.lng != null).length;
const withHours = enriched.filter((e) => (e.live?.openingPeriods?.length ?? 0) > 0).length;
console.log(`enriched: ${enriched.length} total | ${withCoords} with coords | ${withHours} with opening hours`);
console.table(enriched.slice(0, 10).map((e) => ({
  name: e.name,
  type: e.itemType,
  coords: e.lat != null ? `${e.lat.toFixed(3)},${e.lng!.toFixed(3)}` : "(none)",
  hours: e.live?.openingPeriods?.length ?? 0,
  address: e.live?.address?.slice(0, 40) ?? "(none)",
})));


enriched: 30 total | 30 with coords | 20 with opening hours
┌─────────┬──────────────────────────────┬────────────┬──────────────────┬───────┬─────────────────────────────────────────────┐
│ (index) │ name                         │ type       │ coords           │ hours │ address                                     │
├─────────┼──────────────────────────────┼────────────┼──────────────────┼───────┼─────────────────────────────────────────────┤
│ 0       │ 'Kinkaku-ji'                 │ 'activity' │ '35.039,135.729' │ 7     │ '1 Kinkakujichō, Kita Ward, Kyoto, 603-83'  │
│ 1       │ '伏見稻荷大社'               │ 'activity' │ '34.968,135.779' │ 1     │ '68 Fukakusa Yabunouchichō, Fushimi Ward,'  │
│ 2       │ '二條城'                     │ 'activity' │ '35.014,135.748' │ 7     │ '541 Nijōjōchō, Nakagyo Ward, Kyoto, 604-'  │
│ 3       │ 'SAMURAI NINJA MUSEUM Kyoto' │ 'activity' │ '35.007,135.764' │ 7     │ '109 Horinouechō, Nakagyo Ward, Kyoto, 60'  │
│ 4       │ '清水寺'                     │ 'activ

## Phase 3 — LLM pick (Gemini)

Gemini sees only place_id, name, type, tags, and a short summary — never
coords or hours. It returns `{ title, summary, tags, days: [{day_number,
place_ids[]}] }`. The orchestrator filters hallucinated/duplicate IDs and
truncates to the pace cap before returning.

We only ask the LLM to cover days that routes didn't already claim.

In [9]:
let pick = null;

if (compose.uncoveredDays.length === 0) {
  console.log("all days route-covered — skipping LLM. Synthesizing title/summary from routes.");
  pick = __notebook.synthesizePickFromRoutes(compose, answers.destination!);
} else {
  pick = await __notebook.llmPickAssignment(
    input,
    enriched,
    [], // no prior infeasibility issues on first attempt
    undefined, // no prior pick
    {
      onlyDays: compose.uncoveredDays,
      excludePlaceIds: compose.usedPlaceIds,
      genId,
      attempt: 0,
    }
  );
}

console.log("title  :", pick.title);
console.log("summary:", pick.summary);
console.log("tags   :", pick.tags.join(", "));
const byId = new Map(enriched.map((p) => [p.placeId, p]));
for (const d of pick.days) {
  const names = d.place_ids.map((id) => byId.get(id)?.name ?? `(unknown ${id.slice(0,8)})`);
  console.log(`  D${d.day_number}: ${names.join(" → ")}`);
}


{"level":"info","msg":"[gen] phase 3: llm input","ts":"2026-05-26T04:32:02.564Z","genId":"1559d757","attempt":0,"onlyDays":"1,2,3","shortlistSize":30,"shortlistByType":"activity=10,restaurant=10,hotel=10","repairIssues":"(none)","hadPrior":false}
[gemini] calling gemini-2.5-flash with JSON output...
[gemini] raw response length: 1024


{"level":"error","msg":"[gen] phase 3: llm schema invalid","ts":"2026-05-26T04:32:17.388Z","genId":"1559d757","attempt":0,"error":"[\n  {\n    \"origin\": \"array\",\n    \"code\": \"too_big\",\n    \"maximum\": 8,\n    \"inclusive\": true,\n    \"path\": [\n      \"tags\"\n    ],\n    \"message\": \"Too big: expected array to have <=8 items\"\n  }\n]"}
GenerationFailedError: [
  {
    "origin": "array",
    "code": "too_big",
    "maximum": 8,
    "inclusive": true,
    "path": [
      "tags"
    ],
    "message": "Too big: expected array to have <=8 items"
  }
]
    at Object.llmPickAssignment (E:\Projects\travel-sync-ai\services\trip-generation\orchestrator.ts:411:11)
    at process.processTicksAndRejections (node:internal/process/task_queues:105:5)
    at async evalmachine.<anonymous>:10:27
    at async Object.execute (C:\Users\dougl\AppData\Roaming\npm\node_modules\tslab\dist\executor.js:175:17)
    at async JupyterHandlerImpl.handleExecuteImpl (C:\Users\dougl\AppData\Roaming\npm\

### (Optional) Hand-edit the LLM pick

If you want to override the assignment before the solver runs, mutate
`pick.days` here. Example: force a different place_id on day 2.

```ts
// pick!.days = pick!.days.map(d => d.day_number === 2
//   ? { ...d, place_ids: [enriched[0].placeId, enriched[3].placeId] }
//   : d
// );
```

## Phase 4 — Solver

Brute-force permutes each day's stops (≤6 → ≤720 permutations), simulates
travel via Haversine, enforces opening hours and meal anchors (lunch
12–14, dinner 18–20). Returns either `feasible` with timed stops or
`infeasible` with issues — in the real pipeline, infeasibility triggers a
repair loop (LLM swap + route demotion, ≤2 attempts).

In [ ]:
const solved = __notebook.trySolve(pick, enriched, input, startWeekday, compose);

if (solved.kind === "feasible") {
  console.log("FEASIBLE\n");
  for (const day of solved.days) {
    console.log(`Day ${day.dayNumber}`);
    for (const s of day.stops) {
      const arrive = `${String(Math.floor(s.arriveMinutes/60)).padStart(2,"0")}:${String(s.arriveMinutes%60).padStart(2,"0")}`;
      const depart = `${String(Math.floor(s.departMinutes/60)).padStart(2,"0")}:${String(s.departMinutes%60).padStart(2,"0")}`;
      console.log(`  ${arrive}–${depart}  ${s.poi.name}  [${s.poi.itemType}]`);
    }
    console.log("");
  }
} else {
  console.log("INFEASIBLE — issues:");
  for (const issue of solved.issues) {
    console.log(`  D${issue.dayNumber} / ${issue.reason}: ${issue.detail}`);
    if (issue.offendingPlaceIds.length) {
      const names = issue.offendingPlaceIds.map((id) => byId.get(id)?.name ?? id.slice(0,8));
      console.log(`    offending: ${names.join(", ")}`);
    }
  }
  console.log("\nTo simulate the repair loop, re-run Phase 3 with these issues passed in as the 3rd arg of llmPickAssignment, and `pick` as the 4th arg.");
}


### (Optional) Manual repair attempt

Uncomment to re-ask the LLM with the infeasibility report. The orchestrator
does this automatically up to `MAX_REPAIR_ATTEMPTS = 2` times.

```ts
// if (solved.kind === "infeasible" && pick) {
//   const repairedPick = await __notebook.llmPickAssignment(
//     input,
//     enriched,
//     solved.issues.filter((i) => !compose.coveredDays.has(i.dayNumber)),
//     pick,
//     { onlyDays: compose.uncoveredDays, excludePlaceIds: compose.usedPlaceIds, genId, attempt: 1 }
//   );
//   const resolved = __notebook.trySolve(repairedPick, enriched, input, startWeekday, compose);
//   console.log("after repair:", resolved.kind);
// }
```

## Phase 5 — Persist (commented out)

Writes a real `trip_templates` + `trip_template_versions` +
`trip_template_items` row to Supabase. **Uncomment only when you want a
real artifact.**

In [ ]:
// if (solved.kind === "feasible" && pick) {
//   const out = await __notebook.persistTemplate(input, pick, solved.days, enriched, compose);
//   console.log("persisted:", out);
// } else {
//   console.log("skipping persist — solver was not feasible.");
// }
